<a href="https://colab.research.google.com/github/harshs-data/Pytorch/blob/main/textsummarizationT5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install transformers[torch]

In [6]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.0 MB/s eta 0:00:00


The `load_metric` function from the `datasets` library has been deprecated. The new recommended way to load metrics is using the `evaluate` library. I'll update your import cell accordingly.

In [7]:
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
import numpy as np
import evaluate

Now that `evaluate` is imported, you can load a metric like ROUGE, which is commonly used for summarization tasks, as follows:

In [8]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=f718c9f4bb3dd31e5d6de648d5457f8c84f3abc2c2dddaf485c4a9c29fec2ff1
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [9]:
rouge = evaluate.load('rouge')
print(rouge)

EvaluationModule(name: "rouge", module_type: "metric", features: [{'predictions': Value('string'), 'references': List(Value('string'))}, {'predictions': Value('string'), 'references': Value('string')}], usage: """
Calculates average rouge scores for a list of hypotheses and references
Args:
    predictions: list of predictions to score. Each prediction
        should be a string with tokens separated by spaces.
    references: list of reference for each prediction. Each
        reference should be a string with tokens separated by spaces.
    rouge_types: A list of rouge types to calculate.
        Valid names:
        `"rouge{n}"` (e.g. `"rouge1"`, `"rouge2"`) where: {n} is the n-gram based scoring,
        `"rougeL"`: Longest common subsequence based scoring.
        `"rougeLsum"`: rougeLsum splits text using `"
"`.
        See details in https://github.com/huggingface/datasets/issues/617
    use_stemmer: Bool indicating whether Porter stemmer should be used to strip word suffixes.
 

In [10]:
from datasets import load_dataset

dataset = load_dataset("Awesome075/multi_news_parquet")

print(dataset)

README.md:   0%|          | 0.00/2.00k [00:00<?, ?B/s]

train.parquet: reconstructing file:   0%|          |  0.00B /  323MB            

train.parquet: downloading bytes:           |  0.00B            

validation.parquet: reconstructing file:   0%|          |  0.00B / 39.5MB            

validation.parquet: downloading bytes:           |  0.00B            

...

test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/44972 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5622 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5622 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['document', 'summary'],
        num_rows: 44972
    })
    validation: Dataset({
        features: ['document', 'summary'],
        num_rows: 5622
    })
    test: Dataset({
        features: ['document', 'summary'],
        num_rows: 5622
    })
})


In [11]:
print(dataset["train"][0])

{'document': 'National Archives \n \n Yes, it’s that time again, folks. It’s the first Friday of the month, when for one ever-so-brief moment the interests of Wall Street, Washington and Main Street are all aligned on one thing: Jobs. \n \n A fresh update on the U.S. employment situation for January hits the wires at 8:30 a.m. New York time offering one of the most important snapshots on how the economy fared during the previous month. Expectations are for 203,000 new jobs to be created, according to economists polled by Dow Jones Newswires, compared to 227,000 jobs added in February. The unemployment rate is expected to hold steady at 8.3%. \n \n Here at MarketBeat HQ, we’ll be offering color commentary before and after the data crosses the wires. Feel free to weigh-in yourself, via the comments section. And while you’re here, why don’t you sign up to follow us on Twitter. \n \n Enjoy the show. ||||| Employers pulled back sharply on hiring last month, a reminder that the U.S. economy 

In [12]:
dataset['train'].features

{'document': Value('string'), 'summary': Value('string')}

In [13]:
train_subset = dataset["train"].select(range(10000))
validation_subset = dataset["validation"].select(range(1000))
test_subset = dataset["test"].select(range(1000))

In [14]:
checkpoint = 't5-small'
tokenizer = T5Tokenizer.from_pretrained(checkpoint)


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [15]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [16]:
def preprocess_function(examples):
    inputs = ["summarize: " + doc for doc in examples["document"]]

    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True
    )

    labels = tokenizer(
        examples["summary"],
        max_length=150,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [17]:
tokenized_train = train_subset.map(
    preprocess_function,
    batched=True
)

tokenized_validation = validation_subset.map(
    preprocess_function,
    batched=True
)

tokenized_test = test_subset.map(
    preprocess_function,
    batched=True
)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [18]:
model = T5ForConditionalGeneration.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [19]:
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [20]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model
)

In [25]:
import numpy as np
import evaluate

rouge = evaluate.load("rouge")

def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions

    # Decode the predictions and labels
    pred_str = tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    labels_ids[labels_ids == -100] = tokenizer.pad_token_id

    label_str = tokenizer.batch_decode(
        labels_ids,
        skip_special_tokens=True
    )

    # Compute ROUGE scores
    rouge_output = rouge.compute(
        predictions=pred_str,
        references=label_str,
        use_stemmer=True
    )

    # Aggregate the ROUGE scores
    result = {
        key: value * 100  # Corrected: Directly use 'value' as it's already the float score
        for key, value in rouge_output.items()
    }

    # Calculate average generated sequence length
    prediction_lens = [
        np.count_nonzero(pred != tokenizer.pad_token_id)
        for pred in pred_ids
    ]

    result["gen_len"] = np.mean(prediction_lens)

    # Round the results
    result = {
        k: round(v, 4)
        for k, v in result.items()
    }

    return result

In [22]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    generation_max_length=150,
    generation_num_beams=4,
)

In [23]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [26]:
trainer.train()

Epoch,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss


AttributeError: 'numpy.float64' object has no attribute 'mid'

In [ ]:
trainer.evaluate()

test_resluts = trainer.evaluate(eval_dataset=tokenizer_test)

print(test_results)

In [ ]:
import torch

test_index = 0
example_text = dataset["test"][test_index]["document"]

input_text = "summarize: " + example_text

inputs = tokenizer.encode(
    input_text,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

inputs = inputs.to(device)

summary_ids = model.generate(
    inputs,
    max_length=150,
    min_length=40,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("Original Text:\n", example_text)
print("\nGenerated Summary:\n", summary)